# Temporal Deep Learning for Crop Yield Prediction Using Real Data

This notebook is the Colab entry point. The numerical pipeline and model settings are preserved; code is organized into clear stages for reproducibility and maintenance.

In [ ]:
!pip -q install -r requirements.txt
import os, sys, zipfile, glob, requests, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('.'))
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
warnings.filterwarnings('ignore')
np.random.seed(42); tf.keras.utils.set_random_seed(42)
DATA_DIR='data/raw'; PROCESSED_DIR='data/processed'; RESULTS_DIR='results/predictions'; FIGURES_DIR='results/figures'; MODELS_DIR='models'
for path in [DATA_DIR,PROCESSED_DIR,RESULTS_DIR,FIGURES_DIR,MODELS_DIR]: os.makedirs(path,exist_ok=True)


## 1. Download and prepare FAOSTAT crop-yield data

In [ ]:
URL='https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip'
ZIP=os.path.join(DATA_DIR,'faostat_crops.zip'); OUT=os.path.join(DATA_DIR,'faostat_data')
if not os.path.exists(ZIP):
    r=requests.get(URL,timeout=180); r.raise_for_status(); open(ZIP,'wb').write(r.content)
os.makedirs(OUT,exist_ok=True)
with zipfile.ZipFile(ZIP) as z: z.extractall(OUT)
files=glob.glob(OUT+'/**/*.csv',recursive=True); crop_file=next(f for f in files if 'All_Data' in os.path.basename(f))
raw=pd.read_csv(crop_file,encoding='latin-1')
india=raw[raw['Area'].eq('India')].copy(); major=['Wheat','Rice','Maize']
d=india[india['Item'].isin(major) & india['Element'].astype(str).str.lower().str.contains('yield')].copy()
d=d.rename(columns={'Year':'year','Item':'crop','Value':'yield_raw','Unit':'yield_unit'})[['year','crop','yield_raw','yield_unit']]
d['year']=pd.to_numeric(d['year'],errors='coerce'); d['yield_raw']=pd.to_numeric(d['yield_raw'],errors='coerce'); d=d.dropna()
d['yield_tonnes_per_ha']=np.where(d['yield_unit'].astype(str).str.contains('hg/ha',case=False,na=False),d['yield_raw']/10000.0,d['yield_raw'])
d=d.sort_values(['crop','year']).reset_index(drop=True)
print(d.groupby('crop')['yield_tonnes_per_ha'].agg(['count','min','max','mean']))


## 2. Download real environmental data from NASA POWER

In [ ]:
LAT,LON=22.9734,78.6569; start_year,end_year=int(d.year.min()),int(d.year.max())
parameters='T2M,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN'; API='https://power.larc.nasa.gov/api/temporal/daily/point'
frames=[]
for year in range(start_year,end_year+1):
    try:
        q={'parameters':parameters,'community':'AG','longitude':LON,'latitude':LAT,'start':f'{year}0101','end':f'{year}1231','format':'JSON'}
        response=requests.get(API,params=q,timeout=180); response.raise_for_status()
        p=response.json().get('properties',{}).get('parameter')
        if p:
            z=pd.DataFrame(p); z.index=pd.to_datetime(z.index.astype(str),format='%Y%m%d',errors='coerce'); frames.append(z[z.index.notna()].replace(-999,np.nan))
    except Exception as e: print('Skipped',year,e)
if not frames: raise RuntimeError('No NASA POWER data downloaded')
w=pd.concat(frames).apply(pd.to_numeric,errors='coerce').sort_index()
a=w.resample('YE').agg({'T2M':'mean','PRECTOTCORR':'sum','RH2M':'mean','ALLSKY_SFC_SW_DWN':'mean'}).reset_index()
a['year']=a['index'].dt.year; a=a.drop(columns='index').dropna()


## 3. Merge data and create temporal sequences

In [ ]:
df=d[['year','crop','yield_tonnes_per_ha']].merge(a,on='year',how='inner').sort_values(['crop','year']).reset_index(drop=True)
env_features=['T2M','PRECTOTCORR','RH2M','ALLSKY_SFC_SW_DWN']; WINDOW=3; X=[]; y=[]; yrs=[]; crop_ids=[]
crop_to_id={crop:i for i,crop in enumerate(sorted(df.crop.unique()))}
for crop,p in df.groupby('crop'):
    p=p.sort_values('year').reset_index(drop=True)
    for i in range(WINDOW,len(p)):
        hist_years=p.loc[i-WINDOW:i-1,'year'].to_numpy()
        if not np.all(np.diff(hist_years)==1): continue
        X.append(p.loc[i-WINDOW:i-1,env_features].to_numpy(dtype=float)); y.append(float(p.loc[i,'yield_tonnes_per_ha']))
        yrs.append(int(p.loc[i,'year'])); crop_ids.append(crop_to_id[crop])
X=np.asarray(X,dtype=np.float32); y=np.asarray(y,dtype=np.float32); yrs=np.asarray(yrs); crop_ids=np.asarray(crop_ids)
if len(X)<20: raise RuntimeError(f'Only {len(X)} valid temporal samples were created.')
crop_oh=np.eye(len(crop_to_id),dtype=np.float32)[crop_ids]; crop_seq=np.repeat(crop_oh[:,None,:],WINDOW,axis=1); X=np.concatenate([X,crop_seq],axis=2)
split_year=np.sort(np.unique(yrs))[max(1,int(len(np.unique(yrs))*0.8))-1]; tr=yrs<=split_year; te=yrs>split_year
if te.sum()<2:
    order=np.argsort(yrs); cut=max(2,int(len(X)*0.8)); tr=np.zeros(len(X),dtype=bool); tr[order[:cut]]=True; te=~tr
steps,nf=X.shape[1],X.shape[2]; x_scaler=StandardScaler()
Xtr=x_scaler.fit_transform(X[tr].reshape(-1,nf)).reshape(tr.sum(),steps,nf); Xte=x_scaler.transform(X[te].reshape(-1,nf)).reshape(te.sum(),steps,nf)
ytr_raw,yte_raw=y[tr],y[te]; y_scaler=StandardScaler(); ytr=y_scaler.fit_transform(ytr_raw.reshape(-1,1)).ravel().astype(np.float32); yte=y_scaler.transform(yte_raw.reshape(-1,1)).ravel().astype(np.float32)
print('Train:',Xtr.shape,'Test:',Xte.shape,'Yield range:',(ytr_raw.min(),ytr_raw.max()))


## 4. Train and evaluate models

In [ ]:
if len(Xtr)<8 or len(Xte)<2: raise RuntimeError(f'Insufficient split: train={len(Xtr)}, test={len(Xte)}')
tf.keras.backend.clear_session(); model=Sequential([Input(shape=(steps,nf)),LSTM(16,dropout=0.10,recurrent_dropout=0.0,kernel_regularizer=tf.keras.regularizers.l2(1e-4)),Dense(8,activation='relu',kernel_regularizer=tf.keras.regularizers.l2(1e-4)),Dropout(0.10),Dense(1)])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4,clipnorm=1.0),loss='huber',metrics=['mae'])
val_n=max(2,int(len(Xtr)*0.2)); Xfit,Xval=Xtr[:-val_n],Xtr[-val_n:]; yfit,yval=ytr[:-val_n],ytr[-val_n:]
callbacks=[EarlyStopping(monitor='val_loss',patience=30,restore_best_weights=True,min_delta=1e-4),ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=10,min_lr=1e-5)]
history=model.fit(Xfit,yfit,validation_data=(Xval,yval),epochs=300,batch_size=min(8,len(Xfit)),verbose=1,shuffle=False,callbacks=callbacks)
pred_scaled=model.predict(Xte,verbose=0).reshape(-1); pred=y_scaler.inverse_transform(pred_scaled.reshape(-1,1)).ravel()
lo,hi=np.percentile(ytr_raw,[1,99]); margin=max(0.25*(hi-lo),0.1); pred=np.clip(pred,lo-margin,hi+margin)
lstm_metrics={'MAE':mean_absolute_error(yte_raw,pred),'RMSE':np.sqrt(mean_squared_error(yte_raw,pred)),'R2':r2_score(yte_raw,pred)}
ridge=Ridge(alpha=10.0).fit(Xtr.reshape(len(Xtr),-1),ytr_raw); ridge_pred=ridge.predict(Xte.reshape(len(Xte),-1))
ridge_metrics={'MAE':mean_absolute_error(yte_raw,ridge_pred),'RMSE':np.sqrt(mean_squared_error(yte_raw,ridge_pred)),'R2':r2_score(yte_raw,ridge_pred)}
print('LSTM:',lstm_metrics); print('Ridge:',ridge_metrics)
if ridge_metrics['MAE']<lstm_metrics['MAE']: final_pred,final_name=ridge_pred,'Ridge baseline'
else: final_pred,final_name=pred,'LSTM'
results=pd.DataFrame({'year':yrs[te],'actual_yield':yte_raw,'predicted_yield':final_pred}).sort_values('year')
results.to_csv(os.path.join(RESULTS_DIR,'model_predictions.csv'),index=False); model.save(os.path.join(MODELS_DIR,'crop_yield_lstm.keras'))
display(results)
plt.figure(figsize=(9,4)); plt.plot(results['year'],results['actual_yield'],marker='o',label='Actual'); plt.plot(results['year'],results['predicted_yield'],marker='x',label=final_name); plt.xlabel('Year'); plt.ylabel('Yield (tonnes/hectare)'); plt.title(f'{final_name}: Actual vs Predicted Crop Yield'); plt.grid(); plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'actual_vs_predicted.png'),dpi=150); plt.show()
